In [1]:
import pandas as pd

In [2]:
pd.read_csv("processed-data/defaulter.csv")

,receita_cliente,anuidade_emprestimo,anos_casa_propria,telefone_trab,avaliacao_cidade,score_1,score_2,score_3,score_social,troca_telefone,inadimplente
0,16855.246324,2997.000000,12.157324,0,2.0,0.501213,0.003109,0.513171,0.117428,243.0,1
1,13500.000000,2776.050000,12.157324,0,2.0,0.501213,0.269730,0.513171,0.097900,617.0,0
2,11250.000000,2722.188351,12.157324,0,3.0,0.701396,0.518625,0.700184,0.118600,9.0,0
3,27000.000000,6750.000000,3.000000,0,2.0,0.501213,0.649571,0.513171,0.047400,300.0,0
4,22500.000000,3097.800000,12.157324,0,2.0,0.440744,0.509677,0.513171,0.014400,2913.0,1
...,...,...,...,...,...,...,...,...,...,...,...
14573,11250.000000,1893.150000,12.157324,1,2.0,0.501213,0.591424,0.513171,0.117428,545.0,0
14574,33750.000000,4900.050000,17.000000,0,3.0,0.501213,0.563311,0.513171,0.016500,502.0,1
14575,38070.000000,2878.650000,12.157324,0,3.0,0.501213,0.748159,0.513171,0.070100,699.0,0
14576,16855.246324,5154.300000,7.000000,0,2.0,0.501213,0.559936,0.513171,0.030600,1323.0,0


In [3]:
dados=pd.read_csv("processed-data/defaulter.csv")

In [4]:
dados.head()

,receita_cliente,anuidade_emprestimo,anos_casa_propria,telefone_trab,avaliacao_cidade,score_1,score_2,score_3,score_social,troca_telefone,inadimplente
0,16855.246324,2997.000000,12.157324,0,2.0,0.501213,0.003109,0.513171,0.117428,243.0,1
1,13500.000000,2776.050000,12.157324,0,2.0,0.501213,0.269730,0.513171,0.097900,617.0,0
2,11250.000000,2722.188351,12.157324,0,3.0,0.701396,0.518625,0.700184,0.118600,9.0,0
3,27000.000000,6750.000000,3.000000,0,2.0,0.501213,0.649571,0.513171,0.047400,300.0,0
4,22500.000000,3097.800000,12.157324,0,2.0,0.440744,0.509677,0.513171,0.014400,2913.0,1


In [5]:
dados.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14578 entries, 0 to 14577
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   receita_cliente      14578 non-null  float64
 1   anuidade_emprestimo  14578 non-null  float64
 2   anos_casa_propria    14578 non-null  float64
 3   telefone_trab        14578 non-null  int64  
 4   avaliacao_cidade     14578 non-null  float64
 5   score_1              14578 non-null  float64
 6   score_2              14578 non-null  float64
 7   score_3              14578 non-null  float64
 8   score_social         14578 non-null  float64
 9   troca_telefone       14578 non-null  float64
 10  inadimplente         14578 non-null  int64  
dtypes: float64(9), int64(2)
memory usage: 1.2 MB


In [6]:
round(dados["inadimplente"].value_counts(normalize=True)*100, 2)

inadimplente
0    67.65
1    32.35
Name: proportion, dtype: float64

In [7]:
x=dados.drop("inadimplente", axis=1)
y=dados["inadimplente"]

In [8]:
from sklearn.model_selection import train_test_split
RANDOM_STATE=42
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.33, random_state=RANDOM_STATE, stratify=y)

In [9]:
from sklearn.tree import DecisionTreeClassifier

In [10]:
modelo_decision_tree = DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE)
modelo_decision_tree.fit(x_train, y_train)

DecisionTreeClassifier(max_depth=3, random_state=42)

In [11]:
from sklearn.metrics import recall_score
recall_decision_tree = recall_score(y_test, modelo_decision_tree.predict(x_test))
print(f"Recall do DT: {recall_decision_tree:.3f}")

Recall do DT: 0.143


In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

In [13]:
logistic_pipeline = make_pipeline(StandardScaler(), LogisticRegression())
logistic_pipeline.fit(x_train, y_train)

Pipeline(steps=[('standardscaler', StandardScaler()),
                ('logisticregression', LogisticRegression())])

In [15]:
recall_logistic_regression = recall_score(y_test, logistic_pipeline.predict(x_test))
print(f"Recall da LR: {recall_logistic_regression:.3f}")

Recall da LR: 0.253


In [19]:
from sklearn.model_selection import GridSearchCV

In [20]:
import numpy as np

param_grid_dt = {
   "criterion": ["gini", "entropy"],
   "max_depth": np.linspace(6, 12, 4, dtype=int),
   "min_samples_split": np.linspace(5, 20, 4, dtype=int),
   "min_samples_leaf": np.linspace(5, 20, 4, dtype=int),
   "max_features": ["sqrt", "log2"],
   "splitter": ["best", "random"]
}

In [22]:
from sklearn.model_selection import StratifiedKFold

cv=StratifiedKFold(shuffle=True, random_state=RANDOM_STATE)

dt_grid_search=GridSearchCV(estimator=DecisionTreeClassifier(random_state=RANDOM_STATE),
             param_grid=param_grid_dt,
             scoring="recall",
             n_jobs=-1,
             cv=cv)

dt_grid_search.fit(x_train, y_train)

GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
             estimator=DecisionTreeClassifier(random_state=42), n_jobs=-1,
             param_grid={'criterion': ['gini', 'entropy'],
                         'max_depth': array([ 6,  8, 10, 12]),
                         'max_features': ['sqrt', 'log2'],
                         'min_samples_leaf': array([ 5, 10, 15, 20]),
                         'min_samples_split': array([ 5, 10, 15, 20]),
                         'splitter': ['best', 'random']},
             scoring='recall')